In [21]:
import os
from datasets import load_dataset

print(os.getcwd())
print(os.listdir())

dataset = load_dataset("csv", data_files="./support_tickets_balanced_800.csv")

D:\ticker triager\ticket_triager\ml
['.ipynb_checkpoints', 'argency_model.ipynb', 'category_model.ipynb', 'csecond.ipynb', 'first.ipynb', 'inference.py', 'results', 'support_tickets.csv', 'support_tickets_400.csv', 'support_tickets_balanced_800.csv', 'synthetic_data.py', 'tickets_test.csv', 'tickets_train.csv', 'ticket_lora_adapter', 'urgency_lora_adapter', '__pycache__']


In [33]:
# dataset["train"] currently holds everything — split it
split_dataset = dataset["train"].train_test_split(test_size=0.15, seed=42)

# Now you have both splits
dataset = split_dataset  # dataset["train"] and dataset["test"]

In [34]:
dataset

DatasetDict({
    train: Dataset({
        features: ['subject', 'body', 'category', 'urgency'],
        num_rows: 680
    })
    test: Dataset({
        features: ['subject', 'body', 'category', 'urgency'],
        num_rows: 120
    })
})

In [22]:
import pandas as pd
df = pd.read_csv("support_tickets_balanced_800.csv")
df.head()

,subject,body,category,urgency
0,App crashes on launch,"Hi, the app closes every time I try to open it...",technical,low
1,File upload fails,"Good day, uploads fail even when I use support...",technical,low
2,Cannot access old account,"Good day, I no longer have access to the email...",account,high
3,Plan downgrade not applied,"Dear customer support, my requested plan downg...",subscription,medium
4,Account locked,"Hi, my account was locked after several login ...",account,high


In [23]:
df.iloc[0].body

'Hi, the app closes every time I try to open it. I can share the relevant order or account details privately. The issue is intermittent, so guidance would be helpful. Thank you for your assistance.'

In [24]:
from huggingface_hub import model_info

info = model_info("distilbert/distilbert-base-uncased")
print(info.id)

distilbert/distilbert-base-uncased


In [26]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)  # 8 categories

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [27]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    modules_to_save=["classifier", "pre_classifier"],  # <-- add this line
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 740,355 || all params: 67,696,134 || trainable%: 1.0936


In [29]:
BASE_MODEL = "distilbert-base-uncased"

In [ ]:
label_list_urgency = ["low", "medium", "high"]
label2id_urgency = {label: i for i, label in enumerate(label_list_urgency)}
id2label_urgency = {i: label for label, i in label2id_urgency.items()}
# labels is the name expected by Hugging Face Trainer
def tokenize_urgency(batch):
    texts = [s + " " + b for s, b in zip(batch["subject"], batch["body"])]
    encoded = tokenizer(texts, truncation=True, padding="max_length", max_length=128)
    encoded["labels"] = [label2id_urgency[u] for u in batch["urgency"]]
    return encoded

tokenized_urgency = dataset.map(tokenize_urgency, batched=True)

model_urgency = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=3, id2label=id2label_urgency, label2id=label2id_urgency
)

Map:   0%|          | 0/680 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [36]:
lora_config_urgency = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    modules_to_save=["classifier", "pre_classifier"],
)
model_urgency = get_peft_model(model_urgency, lora_config_urgency)

trainer_urgency = Trainer(
    model=model_urgency, args=args,
    train_dataset=tokenized_urgency["train"],
    eval_dataset=tokenized_urgency["test"],
    compute_metrics=compute_metrics,
)
trainer_urgency.train()

C:\Users\PRASHANT\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,1.110731
20,1.098206
30,1.078479
40,1.082966
50,1.066585
60,1.061449
70,1.011224
80,0.988093
90,0.848695
100,0.743467


C:\Users\PRASHANT\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\PRASHANT\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=255, training_loss=0.5781714488478268, metrics={'train_runtime': 750.2735, 'train_samples_per_second': 2.719, 'train_steps_per_second': 0.34, 'total_flos': 68719507107840.0, 'train_loss': 0.5781714488478268, 'epoch': 3.0})

In [37]:
model_urgency.save_pretrained("urgency_lora_adapter")
tokenizer.save_pretrained("urgency_lora_adapter")

C:\Users\PRASHANT\AppData\Roaming\Python\Python311\site-packages\peft\utils\other.py:1496: UserWarning: Unable to fetch remote file due to the following error _ssl.c:989: The handshake operation timed out - silently ignoring the lookup for the file config.json in distilbert-base-uncased.
  warnings.warn(
C:\Users\PRASHANT\AppData\Roaming\Python\Python311\site-packages\peft\utils\save_and_load.py:438: UserWarning: Could not find a config file in distilbert-base-uncased - will assume that the vocabulary was not modified.
  warnings.warn(


('urgency_lora_adapter\\tokenizer_config.json',
 'urgency_lora_adapter\\tokenizer.json')

In [39]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

BASE_MODEL = "distilbert-base-uncased"

CATEGORY_MODEL_PATH = "ticket_lora_adapter"
URGENCY_MODEL_PATH = "urgency_lora_adapter"

category_list = [
    "billing", "account", "orders", "shipping",
    "refund", "technical", "product", "subscription",
]
urgency_list = ["low", "medium", "high"]

category_id2label = {i: l for i, l in enumerate(category_list)}
category_label2id = {l: i for i, l in enumerate(category_list)}
urgency_id2label = {i: l for i, l in enumerate(urgency_list)}
urgency_label2id = {l: i for i, l in enumerate(urgency_list)}

# ---- load category model ----
category_tokenizer = AutoTokenizer.from_pretrained(CATEGORY_MODEL_PATH)
category_base = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(category_list),
    id2label=category_id2label, label2id=category_label2id,
)
category_model = PeftModel.from_pretrained(category_base, CATEGORY_MODEL_PATH)
category_model.eval()

# ---- load urgency model ----
urgency_tokenizer = AutoTokenizer.from_pretrained(URGENCY_MODEL_PATH)
urgency_base = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(urgency_list),
    id2label=urgency_id2label, label2id=urgency_label2id,
)
urgency_model = PeftModel.from_pretrained(urgency_base, URGENCY_MODEL_PATH)
urgency_model.eval()


def predict(subject: str, body: str):
    text = f"{subject} {body}"

    cat_inputs = category_tokenizer(text, truncation=True, padding="max_length", max_length=128, return_tensors="pt")
    urg_inputs = urgency_tokenizer(text, truncation=True, padding="max_length", max_length=128, return_tensors="pt")

    with torch.no_grad():
        cat_logits = category_model(**cat_inputs).logits
        cat_probs = torch.softmax(cat_logits, dim=1)
        cat_conf, cat_id = torch.max(cat_probs, dim=1)

        urg_logits = urgency_model(**urg_inputs).logits
        urg_probs = torch.softmax(urg_logits, dim=1)
        urg_conf, urg_id = torch.max(urg_probs, dim=1)

    category = category_model.config.id2label[cat_id.item()]
    urgency = urgency_model.config.id2label[urg_id.item()]

    return category, cat_conf.item(), urgency, urg_conf.item()


sample_tickets = [
    ("Payment failed", "My card was charged twice for the same order and I need a refund."),
    ("Can't login", "I keep entering my password correctly but it says invalid credentials."),
    ("Package never arrived", "Tracking says delivered but I never received my order."),
    ("App keeps crashing", "The mobile app crashes every single time I try to open the dashboard."),
    ("Cancel my subscription", "I want to cancel my subscription before the next billing cycle."),
    ("Refund status unclear","Good day, the order page does not show the current status of my refund."),
]

print(f"{'Subject':28} | {'Category':14} | {'Conf':6} | {'Urgency':8} | {'Conf':6}")
print("-" * 80)
for subject, body in sample_tickets:
    category, cat_conf, urgency, urg_conf = predict(subject, body)
    print(f"{subject:28} | {category:14} | {cat_conf:.2%} | {urgency:8} | {urg_conf:.2%}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Subject                      | Category       | Conf   | Urgency  | Conf  
--------------------------------------------------------------------------------
Payment failed               | billing        | 81.30% | high     | 65.86%
Can't login                  | account        | 99.14% | medium   | 71.87%
Package never arrived        | shipping       | 94.64% | medium   | 68.94%
App keeps crashing           | technical      | 99.09% | medium   | 50.01%
Cancel my subscription       | subscription   | 87.75% | high     | 82.26%
Refund status unclear        | refund         | 99.75% | low      | 48.99%
